# Experiment — Generate and Check Feedback

Feedback can come from a person, an LLM, tool results or explicit Python checks. A model review compares an output with criteria, while a Python check tests only the conditions we wrote. Both can be incomplete; suggestions must be checked against the original task and facts before they guide a revision.

You will use the same fictional library announcement to compare human feedback, a Python presence check and a model review. Then pass supported review feedback into one revision request and check the result. All inputs are embedded. Offline mode uses labelled examples; optional local OpenAI mode makes **two** requests, one review and one revision. There is no automatic retry or external dataset. Run from the top in a fresh kernel.


## 1. Read the evidence and predict

Which fact is missing? Could a reviewer add a room number or registration link? Write your prediction before continuing. Keep these facts unchanged throughout the comparison.


In [ ]:
# These fictional facts are the reference; do not add a room number or link.
facts = 'Reading workshop: Saturday, 10:00, in the library.'
draft = 'Join our reading workshop this Saturday in the library.'
instructions = 'Revise the announcement using only the supplied facts.'


## Configuration and the shared helper

The following cell repeats the **unchanged** `get_completion(system_prompt, user_prompt)` adapter so this notebook works independently. Offline mode uses exact prompt/input matches and labels its supplied responses. An edited prompt has no new offline model response. For fresh responses, install the companion requirements in a standard local Python/Jupyter environment; keep `.env` beside `COMPANION_VERSION` at the extracted package root (or beside this notebook when used alone). Reuse the configuration from Section 2.3. Browser Python stays offline. The live client has a request timeout and no automatic retries.


In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")

In [ ]:
USE_OPENAI = False  # Keep False for browser/offline practice.
if USE_OPENAI:
    configure_live()  # Uses your private local .env; never put the key in a cell.
else:
    print("OFFLINE: supplied examples; not fresh model responses.")


## 2. Compare feedback sources

**User input:** the supplied human feedback below identifies a missing time. An interactive application can pause using `input()` for a person's review; that optional line is commented out so Run All does not hang.

**Python checks:** the presence check identifies missing required terms. It cannot determine whether they have the right meaning.

**Tool results:** a test runner can provide expected and actual results. Section 7.3 demonstrates that source with a Python function.

**Model review:** we will now ask an LLM to compare this same draft with the facts. This is review of its supplied content, not access to hidden model reasoning.


In [ ]:
human_feedback = 'The draft omits the starting time. Include 10:00.'
print("HUMAN FEEDBACK:", human_feedback)
# Optional, in an interactive session: human_feedback = input("Your feedback: ")
required_terms = ["reading workshop", "Saturday", "10:00", "library"]
missing_terms = [term for term in required_terms if term.lower() not in draft.lower()]
print("PYTHON CHECK — missing terms:", missing_terms)


## 3. Request a model review

The supplied criteria ask for evidence and corrections, not a more confident tone. Compare the feedback with the source facts. If a live reviewer invents a detail, do not run the revision unchanged: correct the feedback variable or stop and explain why.


In [ ]:
review_instructions = 'Review the announcement against the supplied facts.\nCheck the workshop, day, time and location.\nIdentify omissions and unsupported claims.\nFor each problem, state the evidence and a specific correction.\nIf no problem is found, say so. Do not invent an issue.'
review_prompt = f"Facts: {facts}\nDraft: {draft}"
register_sample('Review the announcement against the supplied facts.\nCheck the workshop, day, time and location.\nIdentify omissions and unsupported claims.\nFor each problem, state the evidence and a specific correction.\nIf no problem is found, say so. Do not invent an issue.', 'Facts: Reading workshop: Saturday, 10:00, in the library.\nDraft: Join our reading workshop this Saturday in the library.', 'The draft omits the starting time. Include 10:00.')
feedback = get_completion(review_instructions, review_prompt)
print("MODEL REVIEW")
print(feedback)


## 4. Revise using supported feedback

Inspect the feedback before running this cell. Keep only corrections supported by the source facts or the stated requirements. The original draft and facts are passed again; the reviewer is not allowed to supply new facts.


In [ ]:
revision_prompt = f"""
Task: Write a short announcement for library visitors.
Facts: {facts}
Current draft: {draft}
Feedback: {feedback}
Apply supported corrections only.
Return only the revised announcement.
"""
register_sample('Revise the announcement using only the supplied facts.', '\nTask: Write a short announcement for library visitors.\nFacts: Reading workshop: Saturday, 10:00, in the library.\nCurrent draft: Join our reading workshop this Saturday in the library.\nFeedback: The draft omits the starting time. Include 10:00.\nApply supported corrections only.\nReturn only the revised announcement.\n', 'Join our reading workshop this Saturday at 10:00 in the library.')
revision = get_completion(instructions, revision_prompt)
print(revision)


In [ ]:
def check_announcement(text):
    # Narrow presence checks, not proof of correct meaning or complete factuality.
    required = ["reading workshop", "Saturday", "10:00", "library"]
    missing = [value for value in required if value.lower() not in text.lower()]
    return missing

for label, text in [("Before", draft), ("After", revision)]:
    missing = check_announcement(text)
    print(label + ":", text)
    print("Missing required terms:", missing or "none")


## 5. Change, observe and explain

Change one review criterion, keeping the task, facts and draft fixed. Predict the effect before testing. Supplied offline examples do not measure responses to edited prompts; use local live mode only when you want a fresh model experiment.

Check the words and the meaning: does the notice name the event, day, starting time and location correctly? Does it invent facts? A string containing `10:00` could say “The time is not 10:00”, so passing a term check is insufficient. Save your original/revised instruction, results and a short explanation of what the feedback changed.

**Knowledge check:** reject a proposed registration link when no link is provided. A user can request a different tone but cannot turn an absent source detail into a verified fact. The full recorded requests below let you inspect the observable review/revision sequence.


In [ ]:
for index, record in enumerate(request_records, 1):
    print(f"Request {index} — {record['mode']}")
    print("INPUT:", record["user_prompt"])
    print("RESPONSE:", record["response"])
clear_live()
